# Exercise 8.2: Comparing clustering algorithms

From *Programming in High Energy Particle Physics*, Chapter 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch08/ex08_2_solution.ipynb)

Exercise 8.2 Comparing clustering algorithms Compare the anti-\(k_T\), \(k_T\), and Cambridge/Aachen jet clustering algorithms on a sample of simulated events. Plot the jet \(p_T\) and area distributions for each algorithm with \(R = 0.4\) and \(R = 0.8\).

<details><summary>Hint</summary>

Only the algorithm constant changes: fj.kt_algorithm , fj.cambridge_algorithm , fj.antikt_algorithm . Jet areas need a ClusterSequenceArea with an AreaDefinition using ghost particles. Expect anti-\(k_T\) areas to peak sharply at \(\pi R^2\) and the \(k_T\) areas to be broadly distributed.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q fastjet numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
import fastjet as fj
rng=np.random.default_rng(42)

### Generate reproducible toy events

Each event contains three diffuse hard prongs plus a soft underlying event. These are toy particles, not a Pythia sample. Jet areas are measured with explicit ghosts. Keep the cluster sequence alive while accessing its jets.

In [ ]:
def particle(pt,eta,phi):
    return fj.PseudoJet(pt*np.cos(phi),pt*np.sin(phi),pt*np.sinh(eta),pt*np.cosh(eta))
def make_event(rng):
    particles=[]
    for _ in range(3):
        eta0=rng.uniform(-1.2,1.2);phi0=rng.uniform(-np.pi,np.pi)
        for pt in rng.dirichlet(np.ones(20))*rng.uniform(100,250):
            particles.append(particle(pt,eta0+rng.normal(0,0.18),phi0+rng.normal(0,0.18)))
    for _ in range(30):
        particles.append(particle(rng.exponential(1.5),rng.uniform(-2,2),rng.uniform(-np.pi,np.pi)))
    return particles
n_events=12 # increase for smoother distributions
events=[make_event(rng) for _ in range(n_events)]
algorithms={"anti-kT":fj.antikt_algorithm,"kT":fj.kt_algorithm,"Cambridge/Aachen":fj.cambridge_algorithm}
radii=(0.4,0.8)
results={}
for R in radii:
    for name,algorithm in algorithms.items():
        pts=[];areas=[]
        jet_def=fj.JetDefinition(algorithm,R)
        ghost_spec=fj.GhostedAreaSpec(3.0,1,0.05)
        area_def=fj.AreaDefinition(fj.active_area_explicit_ghosts,ghost_spec)
        for event in events:
            cs=fj.ClusterSequenceArea(event,jet_def,area_def)
            jets=fj.sorted_by_pt(cs.inclusive_jets(20.0))
            jets=[j for j in jets if abs(j.eta())<2.0]
            pts.extend(j.pt() for j in jets)
            areas.extend(j.area() for j in jets)
        results[(R,name)]=(np.asarray(pts),np.asarray(areas))
        print(f"R={R}, {name}: {len(pts)} jets, median area={np.median(areas):.2f}, πR²={np.pi*R*R:.2f}")
assert all(len(v[0])>0 for v in results.values())

In [ ]:
fig,axs=plt.subplots(2,2,figsize=(11,8))
for row,R in enumerate(radii):
    for name in algorithms:
        pt,area=results[(R,name)]
        axs[row,0].hist(pt,bins=np.linspace(20,250,25),density=True,histtype="step",label=name)
        axs[row,1].hist(area,bins=np.linspace(0,2.5,26),density=True,histtype="step",label=name)
    axs[row,1].axvline(np.pi*R*R,color="gray",ls="--",label=r"$\pi R^2$")
    axs[row,0].set(xlabel=r"Jet $p_T$ [GeV]",ylabel="Density / bin",title=f"R={R}")
    axs[row,1].set(xlabel=r"Jet area [Δy Δφ]",ylabel="Density / bin",title=f"R={R}")
    axs[row,0].legend(fontsize=8);axs[row,1].legend(fontsize=8)
plt.tight_layout();plt.show()

Anti-kₜ hard jets have comparatively regular areas centered near πR²; kₜ areas are broader because soft particles affect clustering earlier. Cambridge/Aachen lies between them. The pₜ distributions are similar for isolated hard prongs, but overlapping prongs and soft activity make algorithm and R matter. The ghost grid and small toy sample limit precision.